In [ ]:
# Implement a Wikipedia plot scraper using requests + BeautifulSoup (Parsoid HTML for reliable headings)
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote


def _clean_wiki_text(s: str) -> str:
    """Remove citation markers and normalize whitespace."""
    # Remove citation markers like [1], [note 1], [citation needed]
    s = re.sub(r"\[(?:\d+|note\s*\d+|citation needed)\]", "", s, flags=re.IGNORECASE)
    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s


def get_wikipedia_plot(movie_title: str) -> str | None:
    """Return the full Plot section text from a movie's Wikipedia page.

    Approach:
      - Fetch Parsoid-rendered HTML via REST API for consistent section markup.
      - Locate the section whose heading text is exactly 'Plot' (case-insensitive).
      - Extract all <p> text in that section until the next top-level section.

    Returns:
      Plot text as a single cleaned string, or None if the Plot section doesn't exist.

    Raises:
      ValueError: if movie_title is empty, page not found (404), or a disambiguation page is detected.
      requests.HTTPError: for other HTTP errors.
    """

    if not isinstance(movie_title, str) or not movie_title.strip():
        raise ValueError("movie_title must be a non-empty string")

    title = movie_title.strip()

    # Use Parsoid HTML endpoint (more stable than scraping /wiki/ HTML)
    # Docs: https://en.wikipedia.org/api/rest_v1/
    rest_url = f"https://en.wikipedia.org/api/rest_v1/page/html/{quote(title.replace(' ', '_'))}"
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; plot-scraper/1.0; +https://example.com)",
        "Accept": "text/html; charset=utf-8",
    }

    resp = requests.get(rest_url, headers=headers, timeout=30)
    if resp.status_code == 404:
        raise ValueError(f"Wikipedia page not found (404) for title: {movie_title!r}")
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    # Disambiguation guardrail: Parsoid often includes a hatnote/disambig class; also check title text.
    title_text = soup.title.get_text(" ", strip=True) if soup.title else ""
    if "may refer to" in title_text.lower():
        raise ValueError(f"Disambiguation page encountered for {movie_title!r}")

    # In Parsoid HTML, sections are typically <section data-mw-section-id="..."> with a heading inside.
    # We'll search for a heading whose visible text is 'Plot' (case-insensitive).
    plot_section = None

    for sec in soup.find_all("section"):
        # find the first heading within this section
        hdr = sec.find(re.compile(r"^h[1-6]$"))
        if not hdr:
            continue
        hdr_txt = hdr.get_text(" ", strip=True)
        if hdr_txt and hdr_txt.strip().lower() == "plot":
            plot_section = sec
            break

    if plot_section is None:
        return None

    paragraphs = []
    for p in plot_section.find_all("p", recursive=True):
        txt = p.get_text(" ", strip=True)
        if txt:
            paragraphs.append(txt)

    if not paragraphs:
        return None

    plot_text = "\n\n".join(paragraphs)
    plot_text = _clean_wiki_text(plot_text)

    return plot_text or None


# Example usage
example = get_wikipedia_plot("Inception")

if example is None:
    print("No Plot section found (returned None).")
else:
    print(example[:800] + ("..." if len(example) > 800 else ""))
    print("\n\nLength:", len(example))


In 2018, 23 days after Thanos erased half of all life in the universe, [ a ] Carol Danvers rescues Tony Stark and Nebula from deep space. They reunite with the remaining Avengers — Bruce Banner , Steve Rogers , Thor , Natasha Romanoff , and James Rhodes —and Rocket on Earth. Finding Thanos on an uninhabited planet , they plan to use the Infinity Stones to reverse his actions but learn that Thanos has destroyed them. Enraged, Thor decapitates Thanos. Five years later, Scott Lang escapes from the Quantum Realm . [ b ] Reaching the Avengers Compound , he says he experienced only five hours while trapped. Theorizing that the Quantum Realm allows time travel , they ask a reluctant Stark to help them retrieve the Stones from the past to reverse Thanos's present actions. Stark, Rocket, and Banner...


Length: 4100


In [13]:
# Diagnostic: inspect Wikipedia page structure for Plot header candidates
import re
import requests
from bs4 import BeautifulSoup


def debug_plot_header_candidates(movie_title: str, max_candidates: int = 30):
    """Print header/id diagnostics and a sample of top-level section headings.

    Also prints basic response diagnostics to detect when Wikipedia serves unexpected HTML.
    """
    """Print header/id diagnostics and a sample of top-level section headings."""
    title_slug = movie_title.strip().replace(' ', '_')
    url = f"https://en.wikipedia.org/wiki/{title_slug}"
    headers = {"User-Agent": "Mozilla/5.0 (compatible; plot-scraper/1.0; +https://example.com)"}
    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()
    print("Status:", resp.status_code)
    print("Content-Type:", resp.headers.get("Content-Type"))
    print("HTML length:", len(resp.text))

    soup = BeautifulSoup(resp.text, "html.parser")
    print("<title>:", (soup.title.get_text(" ", strip=True) if soup.title else None))
    content = soup.select_one("div.mw-parser-output") or soup

    # collect headline spans and any element ids containing 'plot'
    id_hits = []
    for el in content.find_all(attrs={"id": True}):
        if re.search(r"plot", el.get("id", ""), flags=re.IGNORECASE):
            id_hits.append((el.name, el.get("id"), el.get_text(" ", strip=True)[:80]))

    header_hits = []
    for hdr in content.find_all(["h2", "h3", "h4"]):
        headline = hdr.find("span", class_="mw-headline")
        txt = headline.get_text(" ", strip=True) if headline else hdr.get_text(" ", strip=True)
        if re.search(r"\bplot\b", txt, flags=re.IGNORECASE):
            header_hits.append((hdr.name, txt, hdr.get("id")))

    print("URL:", url)
    print("\nIDs containing 'plot' (first", max_candidates, "):")
    for row in id_hits[:max_candidates]:
        print(" ", row)

    print("\nHeaders containing word 'plot' (first", max_candidates, "):")
    for row in header_hits[:max_candidates]:
        print(" ", row)

    # Also print the main H2 section headings to understand page structure
    h2s = []
    for hdr in content.find_all("h2"):
        headline = hdr.find("span", class_="mw-headline")
        txt = headline.get_text(" ", strip=True) if headline else hdr.get_text(" ", strip=True)
        if txt:
            h2s.append(txt)

    print("\nTop-level <h2> headings:")
    for t in h2s[:50]:
        print(" -", t)

    # Print any headings at all (h2/h3/h4), since some responses may not include h2s
    all_hdrs = []
    for hdr in content.find_all(["h2", "h3", "h4"]):
        headline = hdr.find("span", class_="mw-headline")
        txt = headline.get_text(" ", strip=True) if headline else hdr.get_text(" ", strip=True)
        if txt:
            all_hdrs.append((hdr.name, txt))

    print("\nFirst headings found (h2/h3/h4):")
    for name, txt in all_hdrs[:50]:
        print(f" - {name}: {txt}")


debug_plot_header_candidates("Avengers: Endgame")


Status: 200
Content-Type: text/html; charset=UTF-8
HTML length: 814403
<title>: Avengers: Endgame - Wikipedia
URL: https://en.wikipedia.org/wiki/Avengers:_Endgame

IDs containing 'plot' (first 30 ):

Headers containing word 'plot' (first 30 ):

Top-level <h2> headings:

First headings found (h2/h3/h4):


In [15]:
# Print full plot text and perform simple sanity checks (start/end snippets + save to file)

# Re-fetch to ensure we have the current string in memory
plot_text = get_wikipedia_plot("Avengers: Endgame")

print("Type:", type(plot_text))
print("Length:", None if plot_text is None else len(plot_text))

if plot_text is None:
    print("Returned None (no Plot section found).")
else:
    # Print the FULL text (this may still be visually truncated by the notebook UI,
    # but the underlying string is complete).
    print("\n--- FULL PLOT TEXT START ---\n")
    print(plot_text)
    print("\n--- FULL PLOT TEXT END ---\n")

    # Additional verification: show last 600 chars to confirm completeness
    tail_n = 600
    print(f"Last {tail_n} chars:\n", plot_text[-tail_n:])

    # Save to a local file so you can open it outside the notebook if desired
    out_path = "avengers_endgame_plot.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(plot_text)
    print("Saved to:", out_path)


Type: <class 'str'>
Length: 4100

--- FULL PLOT TEXT START ---

In 2018, 23 days after Thanos erased half of all life in the universe, [ a ] Carol Danvers rescues Tony Stark and Nebula from deep space. They reunite with the remaining Avengers — Bruce Banner , Steve Rogers , Thor , Natasha Romanoff , and James Rhodes —and Rocket on Earth. Finding Thanos on an uninhabited planet , they plan to use the Infinity Stones to reverse his actions but learn that Thanos has destroyed them. Enraged, Thor decapitates Thanos. Five years later, Scott Lang escapes from the Quantum Realm . [ b ] Reaching the Avengers Compound , he says he experienced only five hours while trapped. Theorizing that the Quantum Realm allows time travel , they ask a reluctant Stark to help them retrieve the Stones from the past to reverse Thanos's present actions. Stark, Rocket, and Banner, who has merged his intelligence with the Hulk's strength, build a time machine. Banner says that altering the past does not affect the